# Step 4: mzML検査 → sage DIA解析 → タンパク質マトリクス構築

DIA-MS の mzML ファイルから、タンパク質 × サンプルの定量マトリクスを構築します。
このノートブックは3つの段階から構成されます：

1. **mzML ファイルの検査**（pymzml）: 装置情報・スペクトル数・centroid 化の確認
2. **sage によるDIA解析**: library-free で PSM と LFQ を計算
3. **タンパク質マトリクス構築**: sage の `lfq.tsv` を Gene Symbol × サンプルに集約

**なぜ sage か**: 論文では DIA-NN v1.8.1 が使われていますが、DIA-NN は商用利用に有料ライセンスが必要です。
本書では **MIT ライセンスの sage-proteomics** で代替します（Apple Silicon で 15ファイル約4.5分）。


---

# Part 1: mzML ファイルの検査

`pymzml` で mzML ファイルを走査し、ファイル数・サイズ・MS1/MS2数・m/z 範囲・RT 範囲・centroid 化の有無・DIA の isolation window 数を確認します。

### ライブラリの読み込み

In [ ]:
import os                 # ファイルパス操作
import glob               # パターンマッチによるファイル列挙
import pandas as pd       # データフレーム操作
import pymzml             # mzMLパーサ（MIT ライセンス）

### 設定（パスの定義）

In [ ]:
# Notebook用パス設定（scripts/ からの相対パスに合わせる）
SCRIPT_DIR = os.path.join(os.getcwd(), "../scripts") if not os.path.exists("../scripts") else "../scripts"
PROJECT_DIR = os.path.join(SCRIPT_DIR, "..")
MZML_DIR = os.path.join(PROJECT_DIR, "data", "raw", "raw_mzML")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TABLES_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(TABLES_DIR, "mzml_inventory.csv")

### ヘルパー関数

In [ ]:
def _is_centroided(element) -> bool:
    """spectrum 要素内の cvParam を走査し centroid spectrum (MS:1000127) を探す。

    mzMLでは各スペクトルの centroid/profile 区別は
    `<cvParam accession="MS:1000127" name="centroid spectrum" />` または
    `<cvParam accession="MS:1000128" name="profile spectrum" />` で示される。
    """
    # 名前空間を無視して accession 属性を確認する
    for child in element.iter():
        acc = child.attrib.get("accession", "")
        if acc == "MS:1000127":
            return True
        if acc == "MS:1000128":
            return False
    return False

### mzML検査関数

In [ ]:
def inspect_mzml(path: str) -> dict:
    """1つのmzMLファイルを走査し、メタ情報を辞書で返す。

    pymzml.run.Reader は lazy にスペクトルをイテレートするので、
    1ファイルあたり数十秒〜1分程度で全スペクトルをパースできる。
    """
    filename = os.path.basename(path)
    filesize_mb = round(os.path.getsize(path) / (1024 * 1024), 1)
    print(f"[inspect] {filename} ({filesize_mb} MB) ...", flush=True)

    reader = pymzml.run.Reader(path)

    # 装置情報（mzML header から取得）
    instrument = ""
    try:
        # pymzml は reader.info に OBO情報を格納する
        instrument_list = reader.info.get("referenceable_param_group_list", [])
        if instrument_list:
            instrument = str(instrument_list)[:120]
    except Exception:
        pass

    total_spectra = 0
    ms1_count = 0
    ms2_count = 0
    ms1_mz_min = float("inf")
    ms1_mz_max = float("-inf")
    ms2_mz_min = float("inf")
    ms2_mz_max = float("-inf")
    rt_min = float("inf")
    rt_max = float("-inf")
    centroided_ms1 = None
    centroided_ms2 = None
    isolation_windows = set()

    for spec in reader:
        total_spectra += 1
        ms_level = spec.ms_level
        rt = spec.scan_time_in_minutes() if spec.scan_time_in_minutes() is not None else None
        if rt is not None:
            if rt < rt_min:
                rt_min = rt
            if rt > rt_max:
                rt_max = rt

        if ms_level == 1:
            ms1_count += 1
            if len(spec.peaks("raw")) > 0:
                mz_arr = spec.peaks("raw")[:, 0]
                lo, hi = float(mz_arr.min()), float(mz_arr.max())
                if lo < ms1_mz_min:
                    ms1_mz_min = lo
                if hi > ms1_mz_max:
                    ms1_mz_max = hi
            if centroided_ms1 is None:
                # MS:1000127 = "centroid spectrum", MS:1000128 = "profile spectrum"
                # cvParam 子要素の accession を走査する
                centroided_ms1 = _is_centroided(spec.element)

        elif ms_level == 2:
            ms2_count += 1
            if len(spec.peaks("raw")) > 0:
                mz_arr = spec.peaks("raw")[:, 0]
                lo, hi = float(mz_arr.min()), float(mz_arr.max())
                if lo < ms2_mz_min:
                    ms2_mz_min = lo
                if hi > ms2_mz_max:
                    ms2_mz_max = hi
            if centroided_ms2 is None:
                centroided_ms2 = _is_centroided(spec.element)

            # Isolation window 幅（DIA判定用）
            try:
                precursors = spec.selected_precursors
                if precursors:
                    # pymzml は isolation window を直接は出さないが
                    # selected_precursors[0]["mz"] で target m/z を取得可能
                    target_mz = precursors[0].get("mz")
                    if target_mz is not None:
                        isolation_windows.add(round(float(target_mz), 1))
            except Exception:
                pass

    reader.close() if hasattr(reader, "close") else None

    return {
        "file": filename,
        "size_MB": filesize_mb,
        "total_spectra": total_spectra,
        "ms1_count": ms1_count,
        "ms2_count": ms2_count,
        "ms1_mz_min": round(ms1_mz_min, 2) if ms1_mz_min != float("inf") else None,
        "ms1_mz_max": round(ms1_mz_max, 2) if ms1_mz_max != float("-inf") else None,
        "ms2_mz_min": round(ms2_mz_min, 2) if ms2_mz_min != float("inf") else None,
        "ms2_mz_max": round(ms2_mz_max, 2) if ms2_mz_max != float("-inf") else None,
        "rt_min_min": round(rt_min, 2) if rt_min != float("inf") else None,
        "rt_max_min": round(rt_max, 2) if rt_max != float("-inf") else None,
        "centroided_ms1": centroided_ms1,
        "centroided_ms2": centroided_ms2,
        "n_isolation_targets": len(isolation_windows),
    }

### メイン

In [ ]:
def main():
    mzml_files = sorted(glob.glob(os.path.join(MZML_DIR, "*.mzML")))
    if not mzml_files:
        raise SystemExit(f"No mzML files found in {MZML_DIR}")
    print(f"Found {len(mzml_files)} mzML files under {MZML_DIR}")

    records = []
    for path in mzml_files:
        try:
            records.append(inspect_mzml(path))
        except Exception as e:
            print(f"  ! failed: {path}: {e}")
            records.append({"file": os.path.basename(path), "error": str(e)})

    df = pd.DataFrame(records)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nWrote {OUTPUT_CSV}")
    print(df.to_string(index=False))

### 実行

In [ ]:
main()

---

# Part 2: sage による DIA 解析

`scripts/sage_config.json` をもとに `sage` CLI を呼び出し、15 mzML ファイルから PSM と LFQ を一気に計算します。出力は `results/sage_output/results.sage.tsv` と `lfq.tsv`。

### ライブラリの読み込み

In [ ]:
import os
import sys
import glob
import shutil
import subprocess
import time

### パス設定

In [ ]:
# Notebook用パス設定（scripts/ からの相対パスに合わせる）
SCRIPT_DIR = os.path.join(os.getcwd(), "../scripts") if not os.path.exists("../scripts") else "../scripts"
PROJECT_DIR = os.path.join(SCRIPT_DIR, "..")
CONFIG_PATH = os.path.join(SCRIPT_DIR, "sage_config.json")
MZML_DIR = os.path.join(PROJECT_DIR, "data", "raw", "raw_mzML")
FASTA_PATH = os.path.join(PROJECT_DIR, "data", "raw", "human_proteome.fasta")
OUT_DIR = os.path.join(PROJECT_DIR, "results", "sage_output")
LOG_DIR = os.path.join(PROJECT_DIR, "results", "logs")

### メイン処理

In [ ]:
def main():
    # sage コマンドが PATH にあるか確認
    sage_bin = shutil.which("sage")
    if sage_bin is None:
        raise SystemExit(
            "sage コマンドが見つかりません。micromamba run -n crc-proteomics python ... "
            "のように crc-proteomics 環境内で実行してください。"
        )
    print(f"sage binary: {sage_bin}")

    # 入力ファイルの存在確認
    if not os.path.exists(FASTA_PATH):
        raise SystemExit(f"FASTA not found: {FASTA_PATH}")
    mzml_files = sorted(glob.glob(os.path.join(MZML_DIR, "*.mzML")))
    if not mzml_files:
        raise SystemExit(f"No mzML files found in {MZML_DIR}")
    print(f"FASTA: {FASTA_PATH}")
    print(f"mzML files: {len(mzml_files)}")
    for f in mzml_files:
        print(f"  - {os.path.basename(f)}")

    # 出力ディレクトリを作成
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)

    # sage コマンドを組み立て
    cmd = [
        sage_bin,
        "--fasta", FASTA_PATH,
        "--output_directory", OUT_DIR,
        "--disable-telemetry-i-dont-want-to-improve-sage",
        CONFIG_PATH,
    ] + mzml_files

    print("\n===== Running sage =====")
    print(" ".join(cmd[:6] + ["...", f"({len(mzml_files)} mzML files)"]))

    t0 = time.time()
    # stdout/stderr を受け取りつつリアルタイム表示
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
    finally:
        rc = proc.wait()
    elapsed = time.time() - t0

    if rc != 0:
        raise SystemExit(f"sage exited with code {rc}")
    print(f"\n===== sage finished in {elapsed/60:.1f} min =====")
    print(f"Output directory: {OUT_DIR}")

### 実行

In [ ]:
main()

---

# Part 3: タンパク質マトリクス構築

sage の `lfq.tsv`（ペプチドレベル）を、FASTA の GN= フィールドで遺伝子シンボルにマップし、Gene × サンプルのタンパク質定量マトリクスに集約します。結果は `results/protein_matrix_from_sage.csv` に保存され、次の step_05.ipynb の入力になります。

### ライブラリの読み込み

In [ ]:
import os
import re
import pandas as pd

### パス設定

In [ ]:
# Notebook用パス設定（scripts/ からの相対パスに合わせる）
SCRIPT_DIR = os.path.join(os.getcwd(), "../scripts") if not os.path.exists("../scripts") else "../scripts"
PROJECT_DIR = os.path.join(SCRIPT_DIR, "..")
LFQ_TSV = os.path.join(PROJECT_DIR, "results", "sage_output", "lfq.tsv")
FASTA = os.path.join(PROJECT_DIR, "data", "raw", "human_proteome.fasta")
OUT_CSV = os.path.join(PROJECT_DIR, "results", "protein_matrix_from_sage.csv")

### パラメータ

In [ ]:
# ペプチドFDRの閾値。論文は <1% なので 0.01。
Q_VALUE_THRESHOLD = 0.01

### FASTA パース: UniProt ID -> Gene Symbol のマップ

In [ ]:
def build_id_to_gene_map(fasta_path: str) -> dict:
    """FASTAヘッダから UniProt accession -> Gene Symbol の辞書を作る。

    UniProt のヘッダ例:
        >sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens OX=9606 GN=TP53 PE=1 SV=4

    ここから accession = "P04637"、gene symbol = "TP53" を抽出する。
    GN= が存在しないエントリは "Entry Name" の先頭語（例: P53_HUMAN -> P53）を使う。
    """
    id_to_gene = {}
    gn_re = re.compile(r"\bGN=([^\s]+)")
    with open(fasta_path, "r") as f:
        for line in f:
            if not line.startswith(">"):
                continue
            # ヘッダ行: ">sp|ACC|NAME DESC..."
            header = line[1:].rstrip()
            first_space = header.find(" ")
            id_part = header[:first_space] if first_space > 0 else header
            desc_part = header[first_space + 1 :] if first_space > 0 else ""

            # id_part = "sp|ACC|NAME" or "tr|ACC|NAME"
            parts = id_part.split("|")
            if len(parts) >= 3:
                acc = parts[1]
                entry_name = parts[2]  # 例: "P53_HUMAN"
            else:
                acc = id_part
                entry_name = id_part

            # GN= フィールドがあればそれを使う
            m = gn_re.search(desc_part)
            if m:
                gene = m.group(1)
            else:
                # GN= がない場合は Entry Name の先頭語
                gene = entry_name.split("_")[0]

            id_to_gene[acc] = gene
    return id_to_gene

### sage proteins カラムから代表UniProt accession を取り出す

In [ ]:
def extract_primary_acc(proteins_field: str) -> str:
    """sage の proteins カラムは "sp|ACC1|NAME;tr|ACC2|NAME;..." の形式。

    最初の要素の accession（2番目のフィールド）を返す。
    """
    if not isinstance(proteins_field, str) or not proteins_field:
        return ""
    first = proteins_field.split(";")[0]
    parts = first.split("|")
    if len(parts) >= 2:
        return parts[1]
    return first

### メイン処理

In [ ]:
def main():
    # ----- 1. lfq.tsv を読み込み -----
    if not os.path.exists(LFQ_TSV):
        raise SystemExit(f"lfq.tsv not found: {LFQ_TSV} (先にstep_04_dia_analysis_sage.pyを実行してください)")
    print(f"読み込み: {LFQ_TSV}")
    df = pd.read_csv(LFQ_TSV, sep="\t")
    print(f"  {len(df)} rows, {len(df.columns)} columns")
    print(f"  columns: {df.columns.tolist()}")

    # ----- 2. q_value でフィルタ -----
    before = len(df)
    df = df[df["q_value"] < Q_VALUE_THRESHOLD].copy()
    print(f"  q_value < {Q_VALUE_THRESHOLD}: {before} -> {len(df)} peptides")

    # ----- 3. サンプル列（.mzML で終わる列）を特定 -----
    meta_cols = ["peptide", "charge", "proteins", "q_value", "score", "spectral_angle"]
    sample_cols = [c for c in df.columns if c not in meta_cols]
    print(f"  サンプル列: {len(sample_cols)}")
    for c in sample_cols:
        print(f"    - {c}")

    # .mzML サフィックスを除去してサンプル名を整形
    clean_name = lambda c: re.sub(r"\.mzML$", "", c)
    df = df.rename(columns={c: clean_name(c) for c in sample_cols})
    sample_cols = [clean_name(c) for c in sample_cols]

    # ----- 4. 代表 UniProt accession を取り出し -----
    df["primary_acc"] = df["proteins"].apply(extract_primary_acc)

    # ----- 5. FASTA から Gene Symbol マップを構築 -----
    print(f"\nFASTAから Gene Symbol を抽出: {FASTA}")
    id_to_gene = build_id_to_gene_map(FASTA)
    print(f"  {len(id_to_gene)} エントリ")

    # ペプチドごとに gene を割り当て（マップに無ければ accession 自体を使う）
    df["gene"] = df["primary_acc"].map(id_to_gene).fillna(df["primary_acc"])

    # ----- 6. Gene ごとに強度を合計 -----
    # ペプチド強度は log スケールではなく生の強度なので、合計が意味を持つ
    # ※ sage の lfq.tsv の値は peak integration の合計（Sum integration）
    matrix = df.groupby("gene")[sample_cols].sum(min_count=1)
    print(f"\nタンパク質マトリクス: {matrix.shape[0]} proteins × {matrix.shape[1]} samples")

    # 0 の値は NaN に置換（検出されなかった = 欠損値として扱う）
    matrix = matrix.replace(0, pd.NA)

    # 有効値のあるタンパク質のみ残す
    before = len(matrix)
    matrix = matrix.dropna(how="all")
    print(f"  全サンプルで0/欠損のタンパク質を除去: {before} -> {len(matrix)}")

    # ----- 7. 保存 -----
    matrix.index.name = "Protein"
    matrix.to_csv(OUT_CSV)
    print(f"\n保存: {OUT_CSV}")
    print(f"  ヘッダ例: {list(matrix.columns)[:5]}")
    print(f"  先頭行例:")
    print(matrix.head())

### 実行

In [ ]:
main()